> **AI-assisted:** This notebook was created with AI assistance. Review paths and configuration before running in production.

# Extract MVP GeoJSON (single FED + DAs)

Colab notebook to stream-extract one federal electoral district (FED) and its dissemination areas (DAs) from `CRMP-full-data.zip`.

**Drive note:** Team data is under **Shared with me** (`CSCC01 - Group Project/Data/`). Add a shortcut to **My Drive** before running (see Cell 2).

**Interim MVP (current zip):** Yukon (`yt`), FED `60001`. See `docs/Missing_Files.md` — `territories.csv` and most `006` regional profile CSVs are missing from the bundle.

**Output:** `single_fed_das.geojson` with **`DGUID` + geometry only** (no census attributes on features).

**DA names + population:** collected externally after export — run `scripts/collect_yt_da_profiles.py` → `map-mvp/data/yt_da_profiles.json` (StatCan CSV download and/or API).

**After download:** place the GeoJSON in `map-mvp/data/single_fed_das.geojson` in this repo, then refresh profiles with the collect script.

In [ ]:
# ==========================================
# Cell 1: Environment initialization (Colab only)
# ==========================================
try:
    from google.colab import drive  # pyright: ignore[reportMissingImports]

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("[WARN] Not running in Google Colab. Drive mount and browser download are skipped.")
    print("       Open this notebook in Colab (GitHub tab) and Runtime -> Run all.")
else:
    drive.mount("/content/drive")
    print("[OK] Google Drive mounted at /content/drive")

    get_ipython().run_line_magic("pip", "install -q geopandas fiona shapely pandas")  # pyright: ignore[reportUndefinedVariable]
    print("[OK] geopandas, fiona, shapely, pandas installed")

In [ ]:
# ==========================================
# Cell 2: Stream-extract and merge data
# ==========================================
import os
import zipfile

import geopandas as gpd  # pyright: ignore[reportMissingModuleSource]
import pandas as pd  # pyright: ignore[reportMissingImports]

# --- Configuration (adjust if needed) ---
ZIP_FILENAME = "CRMP-full-data.zip"

# Web UI path: Shared with me > CSCC01 - Group Project > Data > CRMP-full-data.zip
# Colab cannot read "Shared with me" directly. Add a shortcut to My Drive first:
#   Right-click "CSCC01 - Group Project" -> Organize -> Add shortcut to Drive -> My Drive
ZIP_PATH_CANDIDATES = [
    "/content/drive/MyDrive/CSCC01 - Group Project/Data/" + ZIP_FILENAME,
    "/content/drive/Shareddrives/CSCC01 - Group Project/Data/" + ZIP_FILENAME,
]


def resolve_zip_path(candidates, filename):
    for path in candidates:
        if os.path.exists(path):
            print(f"[OK] Zip resolved: {path}")
            return path

    print("[ERROR] Zip file not found. Checked:")
    for path in candidates:
        print(f"  - {path}")

    mydrive = "/content/drive/MyDrive"
    if os.path.isdir(mydrive):
        print("\n[INFO] Searching MyDrive for zip (after shortcut)...")
        for root, _, files in os.walk(mydrive):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[OK] Found via search: {found}")
                return found

    print("\n[ACTION REQUIRED]")
    print("  1. drive.google.com -> Shared with me -> CSCC01 - Group Project")
    print("  2. Right-click folder -> Organize -> Add shortcut to Drive -> My Drive")
    print("  3. Colab: remount Drive, then re-run this cell")
    raise FileNotFoundError(filename)


ZIP_PATH = resolve_zip_path(ZIP_PATH_CANDIDATES, ZIP_FILENAME)

import hashlib
import re

# --- MVP target (see docs/Missing_Files.md) ---
# Interim: Yukon — only province with confirmed DA + FED layers in current zip.
# Original plan (blocked): prov="on", fed_target="35001"
prov = "yt"
fed_target = "60001"  # Yukon single federal riding (FEDUID 60xxx)

# Set False once 006_dissemination_areas regional CSVs are in the zip
USE_PLACEHOLDER_POPULATION = True

PROV_TO_DA_CSV_BASENAME = {
    "on": "ontario.csv",
    "qc": "quebec.csv",
    "bc": "bc.csv",
    "ab": "prairies.csv",
    "mb": "prairies.csv",
    "sk": "prairies.csv",
    "ns": "atlantic.csv",
    "nb": "atlantic.csv",
    "nl": "atlantic.csv",
    "pe": "atlantic.csv",
    "nt": "territories.csv",
    "nu": "territories.csv",
    "yt": "territories.csv",
}

BOUNDARY_PROV_RE = re.compile(r"census_boundaries/([a-z]{2})/")


def normalize_zip_entries(zf):
    return [n.replace("\\", "/") for n in zf.namelist() if not n.endswith("/")]


def province_boundary_paths(names, prov_code):
    needle = f"census_boundaries/{prov_code}/"
    return [n for n in names if needle in n]


def list_boundary_provinces(names):
    return sorted({m.group(1) for n in names if (m := BOUNDARY_PROV_RE.search(n))})


def find_fed_gpkg(names, prov_code):
    paths = province_boundary_paths(names, prov_code)
    candidates = [
        n for n in paths
        if n.endswith(".gpkg") and f"{prov_code}_electoral_districts" in n.split("/")[-1]
    ]
    for token in ["2023ro", "2013ro", "2003ro"]:
        matches = [n for n in candidates if f"electoral_districts_{token}" in n]
        if matches:
            return matches[0]
    plain = [n for n in candidates if n.endswith(f"{prov_code}_electoral_districts.gpkg")]
    return plain[0] if plain else (candidates[0] if candidates else None)


def find_da_gpkg(names, prov_code):
    paths = province_boundary_paths(names, prov_code)
    exact = f"{prov_code}_dissemination_areas.gpkg"
    matches = [n for n in paths if n.endswith(exact)]
    return matches[0] if matches else None


def find_da_csv(names, prov_code):
    basename = PROV_TO_DA_CSV_BASENAME[prov_code]
    suffix = f"006_dissemination_areas/{basename}"
    matches = [
        n for n in names
        if n.endswith(suffix) or n.split("/")[-1] == basename
    ]
    matches = [
        n for n in matches
        if "census_profiles" in n and not n.endswith("_geo_index.csv")
    ]
    return matches[0] if matches else None


def placeholder_population(dguid):
    """Deterministic fake DA population (400-700) for frontend dev only."""
    digest = hashlib.md5(str(dguid).encode()).hexdigest()
    return 400 + (int(digest[:6], 16) % 301)


with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zip_names = normalize_zip_entries(zf)
    print(f"[OK] Zip contains {len(zip_names)} files")

    FED_REL = find_fed_gpkg(zip_names, prov)
    DA_REL = find_da_gpkg(zip_names, prov)
    CSV_REL = find_da_csv(zip_names, prov) if not USE_PLACEHOLDER_POPULATION else None

    if FED_REL:
        print(f"[OK] FED gpkg: {FED_REL}")
    else:
        print(f"[ERROR] FED gpkg missing for province '{prov}'")

    if DA_REL:
        print(f"[OK] DA gpkg: {DA_REL}")
    else:
        print(f"[ERROR] DA gpkg missing for province '{prov}'")

    if USE_PLACEHOLDER_POPULATION:
        print("[WARN] USE_PLACEHOLDER_POPULATION=True — skipping census CSV join")
        print("       See docs/Missing_Files.md for missing 006_dissemination_areas/*.csv")
    elif CSV_REL:
        print(f"[OK] DA profile CSV: {CSV_REL}")
    else:
        expected = PROV_TO_DA_CSV_BASENAME[prov]
        print(f"[ERROR] DA profile CSV missing (expected *006_dissemination_areas/{expected})")

    required = [("FED", FED_REL), ("DA", DA_REL)]
    if not USE_PLACEHOLDER_POPULATION:
        required.append(("CSV", CSV_REL))
    missing = [label for label, rel in required if rel is None]
    if missing:
        print(f"\n[ERROR] Available boundary provinces: {list_boundary_provinces(zip_names)}")
        raise FileNotFoundError(f"Could not resolve zip members: {missing}")

ZIP_VFS = f"/vsizip/{ZIP_PATH}/"
fed_gpkg = ZIP_VFS + FED_REL
da_gpkg = ZIP_VFS + DA_REL

print("\n=== Path configuration ===")
print(f"ZIP_PATH      : {ZIP_PATH}")
print(f"FED layer     : {FED_REL}")
print(f"DA layer      : {DA_REL}")
print(f"DA profile CSV: {CSV_REL or '(placeholder mode)'}")
print(f"Target FEDUID : {fed_target} (province={prov})")
print(f"\n[OK] Zip exists ({os.path.getsize(ZIP_PATH) / (1024**3):.2f} GB)")

# Projected CRS for Canada — use for bbox + centroid checks (avoids EPSG:4326 vs 3347 mismatch)
WORK_CRS = "EPSG:3347"

# --- Read target FED boundary ---
print("\n=== Reading target FED boundary ===")
feds = gpd.read_file(fed_gpkg)
print(f"[OK] Loaded {len(feds)} FED polygons; CRS={feds.crs}")

if "FEDUID" not in feds.columns:
    print(f"[ERROR] Column FEDUID missing. Available: {list(feds.columns)}")
    raise KeyError("FEDUID")

feds["FEDUID"] = feds["FEDUID"].astype(str)
fed_match = feds[feds["FEDUID"] == str(fed_target)]
if fed_match.empty:
    sample = sorted(feds["FEDUID"].unique())[:10]
    print(f"[ERROR] FEDUID {fed_target} not found in {FED_REL}")
    print(f"        Sample FEDUID values: {sample} ...")
    raise ValueError(f"FEDUID {fed_target} not found")

fed_proj = fed_match.to_crs(WORK_CRS)
target_fed_geom = fed_proj.geometry.iloc[0]
fed_bbox = target_fed_geom.bounds
print(f"[OK] Target FED {fed_target} in {WORK_CRS}; bbox={fed_bbox}")

# --- Stream-prefilter DAs (bbox must be in the DA layer's native CRS) ---
print("\n=== Streaming DAs inside target FED ===")
da_meta = gpd.read_file(da_gpkg, rows=1)
da_native_crs = da_meta.crs
print(f"[OK] DA layer native CRS: {da_native_crs}")

fed_bbox_native = tuple(
    gpd.GeoSeries([target_fed_geom], crs=WORK_CRS)
    .to_crs(da_native_crs)
    .total_bounds
)
print(f"[OK] FED bbox reprojected to DA CRS: {fed_bbox_native}")

das_in_bbox = gpd.read_file(da_gpkg, bbox=fed_bbox_native)
print(f"[OK] BBox prefilter returned {len(das_in_bbox)} DA polygons")

if len(das_in_bbox) == 0:
    print("[WARN] BBox prefilter returned 0 — loading full provincial DA layer (OK for small provinces)")
    das_in_bbox = gpd.read_file(da_gpkg)
    print(f"[OK] Loaded {len(das_in_bbox)} DA polygons for province '{prov}'")

if "DGUID" not in das_in_bbox.columns:
    print(f"[ERROR] Column DGUID missing in DA layer. Available: {list(das_in_bbox.columns)}")
    raise KeyError("DGUID")

das_proj = das_in_bbox.to_crs(WORK_CRS)
das_in_fed = das_proj[das_proj.geometry.centroid.within(target_fed_geom)].copy()
print(f"[OK] {len(das_in_fed)} DAs have centroids inside FED {fed_target}")
if len(das_in_fed) == 0:
    raise ValueError("No DAs matched the target FED. Check fed_target / province / CRS alignment.")

# --- Attach population (CSV join or placeholder) ---
print("\n=== Population data ===")
das_in_fed["DGUID"] = das_in_fed["DGUID"].astype(str)

if USE_PLACEHOLDER_POPULATION:
    das_in_fed["C1_COUNT_TOTAL"] = das_in_fed["DGUID"].map(placeholder_population)
    final_gdf = das_in_fed[["DGUID", "C1_COUNT_TOTAL", "geometry"]].copy()
    print("[WARN] Using placeholder C1_COUNT_TOTAL (not real 2021 census data)")
    print(f"[OK] Placeholder range: {final_gdf['C1_COUNT_TOTAL'].min()} – {final_gdf['C1_COUNT_TOTAL'].max()}")
else:
    print("Chunk-reading 2021 DA population (CHARACTERISTIC_ID == 1)...")
    USECOLS = ["DGUID", "CHARACTERISTIC_ID", "C1_COUNT_TOTAL"]
    pop_records = []
    rows_scanned = 0
    rows_kept = 0
    target_dguids = set(das_in_fed["DGUID"])

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        with zf.open(CSV_REL) as raw_csv:
            try:
                chunk_iter = pd.read_csv(
                    raw_csv,
                    chunksize=20000,
                    usecols=USECOLS,
                    encoding="utf-8",
                    low_memory=False,
                )
            except UnicodeDecodeError:
                print("[WARN] UTF-8 decode failed; retrying with latin-1")
                raw_csv.seek(0)
                chunk_iter = pd.read_csv(
                    raw_csv,
                    chunksize=20000,
                    usecols=USECOLS,
                    encoding="latin-1",
                    low_memory=False,
                )

            for chunk in chunk_iter:
                rows_scanned += len(chunk)
                filtered = chunk[
                    (chunk["CHARACTERISTIC_ID"] == 1)
                    & (chunk["DGUID"].astype(str).isin(target_dguids))
                ]
                rows_kept += len(filtered)
                if not filtered.empty:
                    pop_records.append(filtered)

    print(f"[OK] Scanned {rows_scanned:,} CSV rows; kept {rows_kept:,} population rows")
    if not pop_records:
        raise ValueError("Empty population extract — check CSV path or set USE_PLACEHOLDER_POPULATION=True")

    pop_df = pd.concat(pop_records, ignore_index=True).drop_duplicates(subset=["DGUID"], keep="first")
    pop_df["DGUID"] = pop_df["DGUID"].astype(str)
    final_gdf = das_in_fed.merge(pop_df[["DGUID", "C1_COUNT_TOTAL"]], on="DGUID", how="left")
    missing_pop = final_gdf["C1_COUNT_TOTAL"].isna().sum()
    if missing_pop:
        print(f"[WARN] {missing_pop} DAs have no population after join")
    else:
        print("[OK] All DAs have C1_COUNT_TOTAL from census CSV")
    final_gdf = final_gdf[["DGUID", "C1_COUNT_TOTAL", "geometry"]]

final_gdf = final_gdf.to_crs(epsg=4326)
print(f"[OK] Reprojected to EPSG:4326; {len(final_gdf)} features ready for export")

In [ ]:
# ==========================================
# Cell 3: Export GeoJSON and download
# ==========================================
import os
import sys

OUTPUT_NAME = "single_fed_das.geojson"
REPO_TARGET = "map-mvp/data/single_fed_das.geojson"

final_gdf.to_file(OUTPUT_NAME, driver="GeoJSON")
size_mb = os.path.getsize(OUTPUT_NAME) / (1024 * 1024)

print("=== Export summary ===")
print(f"File          : {OUTPUT_NAME}")
print(f"Province/FED  : {prov} / {fed_target}")
print(f"Population    : {'PLACEHOLDER' if USE_PLACEHOLDER_POPULATION else 'census CSV'}")
print(f"Features      : {len(final_gdf)}")
print(f"Columns       : {[c for c in final_gdf.columns if c != 'geometry']}")
print(f"File size     : {size_mb:.2f} MB")
print(f"Repo target   : {REPO_TARGET}")

if size_mb >= 5:
    print("[WARN] File is >= 5 MB. Definition of Done expects < 5 MB.")
else:
    print("[OK] File size under 5 MB")

required_cols = {"DGUID", "C1_COUNT_TOTAL"}
if required_cols.issubset(set(final_gdf.columns)):
    print(f"[OK] Required attributes present: {sorted(required_cols)}")
else:
    print(f"[ERROR] Missing columns. Expected {required_cols}, got {list(final_gdf.columns)}")

if "google.colab" in sys.modules:
    from google.colab import files  # pyright: ignore[reportMissingImports]

    files.download(OUTPUT_NAME)
    print("\n[OK] Browser download triggered.")
    print(f"     Move {OUTPUT_NAME} into your local repo at: {REPO_TARGET}")
else:
    print("\n[INFO] Not in Colab — file written locally only (no browser download).")